In [0]:
# Cell 1: Install dependencies
%pip install langgraph>=0.2.0 langchain-openai>=0.2.0 openai databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
# Cell 2: Test if GDPRAgent can be imported and instantiated
import sys
import os
sys.path.append('/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent')

# Set OpenAI API key from Databricks secrets
os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope="openai", key="GDPR_agent")

from gdpr_agent.agent import GDPRAgent

try:
    agent = GDPRAgent()
    print("✅ GDPRAgent instantiated successfully")
    print(f"   Type: {type(agent)}")
except Exception as e:
    print(f"❌ Failed to create GDPRAgent: {e}")
    import traceback
    traceback.print_exc()

In [0]:
# Cell 3: Register model with MLflow wrapper
import mlflow
import os
from gdpr_agent.agent import GDPRAgent

# Ensure OpenAI key is set
os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope="openai", key="GDPR_agent")


class GDPRAgentWrapper(mlflow.pyfunc.PythonModel):
    """MLflow wrapper for GDPRAgent"""
    
    def __init__(self):
        """Initialize without creating agent yet (happens in load_context)"""
        self.agent = None
    
    def load_context(self, context):
        """Called when model is loaded for serving"""
        import os
        # Agent will be initialized when loaded in serving environment
        self.agent = GDPRAgent()
    
    def predict(self, context, model_input):
        """
        Handle inference requests.
        
        Expected input:
        - pandas DataFrame with 'question' column
        - dict with 'question' key
        - list of dicts with 'question' key
        """
        import pandas as pd
        
        # Ensure agent is loaded
        if self.agent is None:
            self.agent = GDPRAgent()
        
        # Handle different input types
        if isinstance(model_input, pd.DataFrame):
            questions = model_input['question'].tolist()
        elif isinstance(model_input, dict):
            questions = [model_input.get('question', '')]
        elif isinstance(model_input, list):
            questions = [q.get('question', '') if isinstance(q, dict) else str(q) for q in model_input]
        else:
            questions = [str(model_input)]
        
        # Process each question
        results = []
        for question in questions:
            try:
                # Call your agent's query method
                response = self.agent.query(question)
                results.append({
                    'answer': response.get('answer', ''),
                    'context': response.get('context', []),
                    'sources': response.get('sources', [])
                })
            except Exception as e:
                results.append({
                    'answer': f"Error: {str(e)}",
                    'context': [],
                    'sources': []
                })
        
        return results


# Register the model
mlflow.set_experiment("/Shared/gdpr-agent-staging")

with mlflow.start_run(run_name="manual_test") as run:
    
    mlflow.log_params({
        "commit_sha": "test123",
        "llm_model": "gpt-4o-mini",
        "deployment_target": "staging"
    })
    
    mlflow.log_metrics({
        "eval_pass_rate": 1.0
    })
    
    # Log the wrapped model
    try:
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=GDPRAgentWrapper(),  # ← Use the wrapper
            registered_model_name="gdpr_agent_staging",
            pip_requirements=[
                "openai>=1.12.0",
                "langgraph>=0.2.0",
                "databricks-vectorsearch",
                "mlflow"
            ]
        )
        print("✅ Model registered successfully!")
        print(f"   Run ID: {run.info.run_id}")
        
    except Exception as e:
        print(f"❌ Failed to register model:")
        print(f"   Error: {e}")
        import traceback
        traceback.print_exc()

In [0]:
# Cell 4a: Restart Python first
dbutils.library.restartPython()

In [0]:
# Test the register_staging.py script
import sys
sys.path.append('/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent')

from deploy.register_staging import register_staging_model

# Test registration
version = register_staging_model(
    commit_sha="test_from_notebook",
    pass_rate=0.95
)

print(f"Registered version: {version}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get endpoint details
endpoint = w.serving_endpoints.get("gdpr-agent-staging")

# Check the state
print(f"Endpoint state: {endpoint.state}")
print(f"Config update: {endpoint.state.config_update if endpoint.state else 'None'}")

# Get logs (if available)
if endpoint.state and endpoint.state.config_update == "UPDATE_FAILED":
    print("\n❌ Deployment failed!")
    print("\nCheck logs in:")
    print("Serving → gdpr-agent-staging → Logs tab")

In [0]:
# Cell 7: Check current endpoint state
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint = w.serving_endpoints.get("gdpr-agent-staging")

print(f"Current state: {endpoint.state}")
print(f"Config update: {endpoint.state.config_update if endpoint.state else 'None'}")
print(f"Ready status: {endpoint.state.ready if endpoint.state else 'None'}")

# Show served entities
if endpoint.config and endpoint.config.served_entities:
    print("\n📦 Served entities:")
    for entity in endpoint.config.served_entities:
        print(f"  - {entity.name}")
        print(f"    Model: {entity.entity_name} v{entity.entity_version}")
        print(f"    Workload: {entity.workload_size}")

# Show all possible states
print(f"\n🔍 Debug - Full endpoint state:")
print(f"  endpoint.state = {endpoint.state}")

In [0]:
# Cell 8: Check served entities details
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint = w.serving_endpoints.get("gdpr-agent-staging")

print("❌ Endpoint deployment FAILED\n")

# Show what model failed to deploy
if endpoint.config and endpoint.config.served_entities:
    print("📦 Attempted to deploy:")
    for entity in endpoint.config.served_entities:
        print(f"  Name: {entity.name}")
        print(f"  Model: {entity.entity_name}")
        print(f"  Version: {entity.entity_version}")
        print(f"  Workload: {entity.workload_size}")
        print()

print("🔍 To see detailed error logs:")
print("1. Go to: Serving → Endpoints → gdpr-agent-staging")
print("2. Click 'Logs' tab")
print("3. Look for error containing 'ModuleNotFoundError' or 'ImportError'")

In [0]:
# Cell 9: Debug endpoint configuration
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint = w.serving_endpoints.get("gdpr-agent-staging")

print("❌ Endpoint deployment FAILED\n")

# Debug - check what's actually in the endpoint
print("🔍 Debug - Endpoint config:")
print(f"  endpoint.config = {endpoint.config}")
print(f"  endpoint.config.served_entities = {endpoint.config.served_entities if endpoint.config else 'None'}")

# Try to show pending config if it exists
if hasattr(endpoint, 'pending_config') and endpoint.pending_config:
    print("\n📦 Pending config (attempted deploy):")
    if endpoint.pending_config.served_entities:
        for entity in endpoint.pending_config.served_entities:
            print(f"  Name: {entity.name}")
            print(f"  Model: {entity.entity_name}")
            print(f"  Version: {entity.entity_version}")

print("\n🔍 To see detailed error logs:")
print("1. Go to: Serving → Endpoints → gdpr-agent-staging")
print("2. Click 'Logs' tab")
print("3. Look for error containing 'ModuleNotFoundError' or 'ImportError'")